# UPI / Fintech Transaction Analysis

**Learning project:** synthetic data, not real customer transactions. A flag is not proof of fraud.

We will load the data, inspect quality, create hour/month features, summarize patterns, and visualize results.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data' / 'upi_transactions.csv'
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Understand the dataset
`shape` gives rows and columns. `info()` shows data types and missing values. `describe()` summarizes numeric columns.

In [ ]:
print('Rows, columns:', df.shape)
df.info()
df.describe(include='all').T

## 2. Data quality checks
We check duplicate transaction IDs, missing values, invalid amounts, and convert timestamp to datetime.

In [ ]:
print('Duplicate IDs:', df['transaction_id'].duplicated().sum())
print('\nMissing values:\n', df.isna().sum())
print('Invalid amounts:', (df['amount_inr'].isna() | (df['amount_inr'] < 0)).sum())
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.drop_duplicates(subset=['transaction_id']).dropna(subset=['timestamp', 'amount_inr']).copy()
df = df[df['amount_inr'] >= 0].copy()
print('Rows after cleaning:', len(df))

## 3. Create useful features
Hour and month help compare when transactions happen. These are derived columns, not original data.

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.to_period('M').astype(str)
df[['timestamp', 'hour', 'month', 'amount_inr']].head()

## 4. SQL-like summaries with Pandas
Group by status and payment method to compare counts and transaction value.

In [ ]:
by_status = df.groupby('transaction_status', as_index=False).agg(transaction_count=('transaction_id','count'), total_amount_inr=('amount_inr','sum'), average_amount_inr=('amount_inr','mean'))
by_status

In [ ]:
by_method = df.groupby('payment_method', as_index=False).agg(transaction_count=('transaction_id','count'), total_amount_inr=('amount_inr','sum')).sort_values('total_amount_inr', ascending=False)
by_method

## 5. Illustrative risk flags
The demo label is rule-generated. It is useful for practicing filtering and aggregation, but it does not represent verified fraud.

In [ ]:
flagged = df[df['is_flagged'] == 1].copy()
print('Flagged demo records:', len(flagged))
print('Flagged share (%):', round(100 * len(flagged) / len(df), 2))
flagged.groupby('hour').size().sort_values(ascending=False).head(10)

## 6. Visualize
Each chart answers a different question. Always describe patterns as observations, not proof of cause.

In [ ]:
plt.figure(figsize=(8,4))
df['transaction_status'].value_counts().plot(kind='bar')
plt.title('Transactions by Status'); plt.xlabel('Status'); plt.ylabel('Count'); plt.tight_layout(); plt.show()

In [ ]:
hourly = df.groupby('hour').size().reset_index(name='transaction_count')
plt.figure(figsize=(9,4))
plt.plot(hourly['hour'], hourly['transaction_count'], marker='o')
plt.xticks(range(0,24,2)); plt.title('Transaction Volume by Hour'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(by_method['payment_method'], by_method['total_amount_inr'])
plt.title('Total Transaction Value by Payment Method'); plt.tight_layout(); plt.show()

## 7. Practice questions
1. Which status has the most transactions?
2. Which payment method has the highest total value?
3. Which hour has the highest transaction count?
4. Why is a flagged transaction not the same as confirmed fraud?
5. How would you validate this approach on real data?